# 🚀 Модуль 12. Урок 1. Концепция агентных рабочих процессов (ReAct Pattern: Reason + Act)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_12_ai_agents/lesson_01_agentic_workflows_concept.ipynb)

## 📌 Бизнес-контекст и методология
AI-агент отличается от обычного чат-бота наличием цикла действий и инструментов (Tool Calling).
Паттерн **ReAct**:
1. **Thought (Размышление)**: агент оценивает текущее состояние.
2. **Action (Действие)**: вызов внешнего инструмента (SQL-запрос, расчет в Python, HTTP API).
3. **Observation (Наблюдение)**: получение результата работы инструмента.
4. Повторение цикла до достижения финального ответа.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# Симуляция цикла ReAct агента
def execute_agent_step(goal, current_step, tool_output=None):
    if current_step == 0:
        return {"thought": "Нужно получить список заказов за прошлую неделю", "action": "run_sql('SELECT count(*) FROM orders')"}
    elif current_step == 1:
        return {"thought": f"Получено наблюдение: {tool_output}. Теперь формирую ответ.", "action": "FINISH", "final_answer": f"Всего заказов: {tool_output}"}

step0 = execute_agent_step("Посчитай заказы", 0)
step1 = execute_agent_step("Посчитай заказы", 1, tool_output="14,250")

print(f"Шаг 1: {step0['thought']} -> {step0['action']}")
print(f"Шаг 2: {step1['thought']} -> {step1['final_answer']}")


Шаг 1: Нужно получить список заказов за прошлую неделю -> run_sql('SELECT count(*) FROM orders')
Шаг 2: Получено наблюдение: 14,250. Теперь формирую ответ. -> Всего заказов: 14,250


### 🎯 Практическое задание 1: Проверка завершения цикла
Убедитесь, что на шаге 1 агент вызвал действие `'FINISH'`.

In [3]:
### ВАШ КОД ЗДЕСЬ
assert step1["action"] == "FINISH"
assert "14,250" in step1["final_answer"]
print("✅ Тест успешно пройден: цикл ReAct отработал штатно!")


✅ Тест успешно пройден: цикл ReAct отработал штатно!


### 💡 Защита от бесконечного цикла
Всегда устанавливайте параметр `max_iterations` (обычно 5-10 шагов), чтобы агент не зациклился при ошибке инструмента.